# EDA narrativo · COVID Tracking Project

**Objetivo:** responder 3 preguntas de negocio con estadística, visualización
e interpretación, usando el dataset limpio.

**Principio:** las preguntas y las hipótesis se escriben **antes** de mirar
los resultados. Si una pregunta no se puede responder, se dice.


## 1. Contexto y datos

- **Fuente:** COVID Tracking Project · States Daily (última foto: 2021-03-07).
- **Granularidad:** una fila por `state` y `date`.
- **Cobertura:** 50 estados + DC + territorios.
- **Limitaciones:** casos reportados (no reales), subregistro, cambios de
  definición de `totalTestResults`, efecto fin de semana.

El dataset limpio se carga desde `src.eda.clean.build_clean_dataset()`.


## 2. Preguntas de negocio e hipótesis

### Pregunta 1 · ¿Qué estados concentran más casos y muertes acumuladas?

- **Hipótesis:** unos pocos estados concentran la mayor parte de los casos y
  de las muertes acumuladas; el ranking por counts favorece a los estados
  grandes.
- **Métrica:** `positive` y `death` acumulados por `state`.
- **Periodo:** todo el dataset (2020-01-13 → 2021-03-07).
- **Criterio de respuesta:** tabla ordenada de los 10 estados con mayor count
  y su porcentaje sobre el total.
- **Limitación conocida:** los counts no son comparables entre estados sin
  normalizar por población (se aborda en N3).

### Pregunta 2 · ¿La positividad se asocia con la mortalidad?

- **Hipótesis:** los estados con mayor positividad (`positive / totalTestResults`)
  también presentan mayor tasa de mortalidad (`death / positive`), por
  diferencias en acceso a tests y en la población atendida.
- **Métrica:** `positivity` y `death_rate` por estado (agregado del último
  día disponible).
- **Periodo:** último día disponible por estado.
- **Criterio de respuesta:** scatter con tendencia y coeficiente de
  correlación, con su limitación.
- **Limitación conocida:** correlación no implica causalidad; ambas variables
  dependen del reporting.

### Pregunta 3 · ¿Cómo evolucionan los casos diarios y qué papel tiene el fin de semana?

- **Hipótesis:** la serie diaria de `positiveIncrease` muestra olas claras y
  caídas sistemáticas en fin de semana por menor reporting.
- **Métrica:** `positiveIncrease` diario (total nacional) y media móvil de 7
  días.
- **Periodo:** todo el dataset.
- **Criterio de respuesta:** serie original + media móvil, con lectura de las
  olas y del efecto fin de semana.
- **Limitación conocida:** la media móvil suaviza picos reales; no predice.

Estas preguntas se responden en las secciones 4 (univariado), 5 (bivariado) y
6 (tendencias).


## 3. Configuración e imports


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.eda.clean import build_clean_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)


## 4. Carga del dataset limpio


In [ ]:
clean, audit = build_clean_dataset()
print("shape:", clean.shape)
print("periodo:", clean["date"].min().date(), "->", clean["date"].max().date())
print("auditoría:", audit)
clean.head(3)


## 5. Perfil rápido


In [ ]:
print("filas:", len(clean), "columnas:", clean.shape[1])
print("estados:", clean["state"].nunique())
print("nulos totales (%):", round(clean.isna().mean().mean() * 100, 2))


## 6. Análisis univariado


## 6. Análisis univariado

Se analizan 3 variables: casos acumulados, muertes acumuladas y casos
diarios. Para cada una se muestra distribución, medidas de centro y
dispersión, y una lectura.


### 6.1 Casos acumulados por estado

**Pregunta:** ¿cómo se distribuyen los casos acumulados entre estados?
**Método:** se toma el último día disponible de cada estado y se describe
la distribución.


In [ ]:
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

ultimo = clean.loc[clean.groupby("state")["date"].idxmax()].copy()
print("estados:", len(ultimo))
print(ultimo["positive"].describe().round(0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ultimo["positive"].plot.hist(bins=15, ax=axes[0], color="#4C78A8", edgecolor="white")
axes[0].set_title("Distribución de casos acumulados por estado")
axes[0].set_xlabel("Casos acumulados")
axes[0].set_ylabel("Número de estados")

ultimo["positive"].plot.box(ax=axes[1], vert=True)
axes[1].set_title("Boxplot de casos acumulados")
axes[1].set_ylabel("Casos acumulados")
fig.tight_layout()
fig.savefig(FIGURES / "univariado_casos_acumulados.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
top10 = ultimo.nlargest(10, "positive")[["state", "positive", "death"]]
top10["pct_del_total"] = (top10["positive"] / ultimo["positive"].sum() * 100).round(1)
top10


**Lectura.** La distribución está muy sesgada a la derecha: unos pocos
estados concentran la mayor parte de los casos. El boxplot confirma varios
valores atípicos por arriba (California, Texas, Florida, Nueva York). Esto
refuerza la hipótesis de la Pregunta 1 y anticipa la necesidad de
normalizar por población (N3).

**Limitación:** se usan counts absolutos; los estados grandes siempre
aparecen arriba.


### 6.2 Muertes acumuladas por estado

**Pregunta:** ¿la distribución de muertes es tan desigual como la de casos?
**Método:** igual que en 6.1, sobre `death`.


In [ ]:
print(ultimo["death"].describe().round(0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ultimo["death"].plot.hist(bins=15, ax=axes[0], color="#E45756", edgecolor="white")
axes[0].set_title("Distribución de muertes acumuladas por estado")
axes[0].set_xlabel("Muertes acumuladas")
axes[0].set_ylabel("Número de estados")

ultimo["death"].plot.box(ax=axes[1], vert=True)
axes[1].set_title("Boxplot de muertes acumuladas")
axes[1].set_ylabel("Muertes acumuladas")
fig.tight_layout()
fig.savefig(FIGURES / "univariado_muertes_acumuladas.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
top10d = ultimo.nlargest(10, "death")[["state", "death"]]
top10d["pct_del_total"] = (top10d["death"] / ultimo["death"].sum() * 100).round(1)
top10d


**Lectura.** La distribución de muertes es todavía más sesgada que la de
casos: la mediana es muy inferior a la media y hay varios estados muy por
encima del resto. Nueva York, California, Texas y Florida vuelven a destacar.

**Limitación:** sin normalizar por población no se sabe si un estado es
"peor" o simplemente más grande.


### 6.3 Casos diarios (positiveIncrease)

**Pregunta:** ¿cómo se distribuyen los casos nuevos por día y estado?
**Método:** distribución de `positiveIncrease` y conteo de valores negativos.


In [ ]:
inc = clean["positiveIncrease"].dropna()
print("n:", len(inc))
print(inc.describe().round(0))
print("valores negativos:", int((inc < 0).sum()))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
inc.clip(lower=0).plot.hist(bins=40, ax=ax, color="#72B7B2", edgecolor="white")
ax.set_title("Distribución de casos diarios nuevos (positiveIncrease)")
ax.set_xlabel("Casos nuevos por día")
ax.set_ylabel("Frecuencia")
fig.tight_layout()
fig.savefig(FIGURES / "univariado_casos_diarios.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** La mayoría de los registros diarios son de pocos cientos de
casos, pero la cola derecha es muy larga (días de picos). Hay 24 valores
negativos, que son correcciones de reporting y no errores de limpieza.

**Limitación:** la serie diaria está dominada por los estados grandes; la
media móvil (N2) la suaviza.


## 6. Análisis univariado

Se analizan 3 variables: casos acumulados, muertes acumuladas y casos
diarios. Para cada una se muestra distribución, medidas de centro y
dispersión, y una lectura.


### 6.1 Casos acumulados por estado

**Pregunta:** ¿cómo se distribuyen los casos acumulados entre estados?
**Método:** se toma el último día disponible de cada estado y se describe
la distribución.


In [ ]:
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

ultimo = clean.loc[clean.groupby("state")["date"].idxmax()].copy()
print("estados:", len(ultimo))
print(ultimo["positive"].describe().round(0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ultimo["positive"].plot.hist(bins=15, ax=axes[0], color="#4C78A8", edgecolor="white")
axes[0].set_title("Distribución de casos acumulados por estado")
axes[0].set_xlabel("Casos acumulados")
axes[0].set_ylabel("Número de estados")

ultimo["positive"].plot.box(ax=axes[1], vert=True)
axes[1].set_title("Boxplot de casos acumulados")
axes[1].set_ylabel("Casos acumulados")
fig.tight_layout()
fig.savefig(FIGURES / "univariado_casos_acumulados.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
top10 = ultimo.nlargest(10, "positive")[["state", "positive", "death"]]
top10["pct_del_total"] = (top10["positive"] / ultimo["positive"].sum() * 100).round(1)
top10


**Lectura.** La distribución está muy sesgada a la derecha: unos pocos
estados concentran la mayor parte de los casos. El boxplot confirma varios
valores atípicos por arriba (California, Texas, Florida, Nueva York). Esto
refuerza la hipótesis de la Pregunta 1 y anticipa la necesidad de
normalizar por población (N3).

**Limitación:** se usan counts absolutos; los estados grandes siempre
aparecen arriba.


### 6.2 Muertes acumuladas por estado

**Pregunta:** ¿la distribución de muertes es tan desigual como la de casos?
**Método:** igual que en 6.1, sobre `death`.


In [ ]:
print(ultimo["death"].describe().round(0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ultimo["death"].plot.hist(bins=15, ax=axes[0], color="#E45756", edgecolor="white")
axes[0].set_title("Distribución de muertes acumuladas por estado")
axes[0].set_xlabel("Muertes acumuladas")
axes[0].set_ylabel("Número de estados")

ultimo["death"].plot.box(ax=axes[1], vert=True)
axes[1].set_title("Boxplot de muertes acumuladas")
axes[1].set_ylabel("Muertes acumuladas")
fig.tight_layout()
fig.savefig(FIGURES / "univariado_muertes_acumuladas.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
top10d = ultimo.nlargest(10, "death")[["state", "death"]]
top10d["pct_del_total"] = (top10d["death"] / ultimo["death"].sum() * 100).round(1)
top10d


**Lectura.** La distribución de muertes es todavía más sesgada que la de
casos: la mediana es muy inferior a la media y hay varios estados muy por
encima del resto. Nueva York, California, Texas y Florida vuelven a destacar.

**Limitación:** sin normalizar por población no se sabe si un estado es
"peor" o simplemente más grande.


### 6.3 Casos diarios (positiveIncrease)

**Pregunta:** ¿cómo se distribuyen los casos nuevos por día y estado?
**Método:** distribución de `positiveIncrease` y conteo de valores negativos.


In [ ]:
inc = clean["positiveIncrease"].dropna()
print("n:", len(inc))
print(inc.describe().round(0))
print("valores negativos:", int((inc < 0).sum()))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
inc.clip(lower=0).plot.hist(bins=40, ax=ax, color="#72B7B2", edgecolor="white")
ax.set_title("Distribución de casos diarios nuevos (positiveIncrease)")
ax.set_xlabel("Casos nuevos por día")
ax.set_ylabel("Frecuencia")
fig.tight_layout()
fig.savefig(FIGURES / "univariado_casos_diarios.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** La mayoría de los registros diarios son de pocos cientos de
casos, pero la cola derecha es muy larga (días de picos). Hay 24 valores
negativos, que son correcciones de reporting y no errores de limpieza.

**Limitación:** la serie diaria está dominada por los estados grandes; la
media móvil (N2) la suaviza.


## 7. Análisis bivariado


## 7. Análisis bivariado

Se analizan 3 relaciones entre variables: tests vs positivos, positividad vs
mortalidad y distribución diaria por estado.


### 7.1 Tests realizados vs positivos acumulados

**Pregunta:** ¿los estados que más testean acumulan más positivos?
**Método:** scatter en escala logarítmica con línea de tendencia y
correlación de Pearson.


In [ ]:
import numpy as np

ultimo = clean.loc[clean.groupby("state")["date"].idxmax()].copy()
ultimo["positivity"] = ultimo["positive"] / ultimo["totalTestResults"]
ultimo["death_rate"] = ultimo["death"] / ultimo["positive"]

r = ultimo[["totalTestResults", "positive"]].corr().iloc[0, 1]
print(f"correlación Pearson (tests, positivos) = {r:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(
    data=ultimo, x="totalTestResults", y="positive", ax=ax,
    scatter_kws={"alpha": 0.7}, line_kws={"color": "#E45756"},
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Tests realizados vs positivos acumulados (escala log)")
ax.set_xlabel("Total de tests (acumulados)")
ax.set_ylabel("Positivos (acumulados)")
fig.tight_layout()
fig.savefig(FIGURES / "bivariado_tests_vs_positivos.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** La relación es casi lineal en escala log: a más tests, más
positivos detectados. La correlación es alta, pero no implica causalidad:
ambas variables crecen con el tamaño del estado y con el tiempo.

**Limitación:** no se controla por población ni por periodo.


### 7.2 Positividad vs tasa de mortalidad

**Pregunta:** ¿los estados con mayor positividad tienen mayor mortalidad?
**Método:** scatter de `positivity` vs `death_rate` con tendencia y
correlación.


In [ ]:
r2 = ultimo[["positivity", "death_rate"]].corr().iloc[0, 1]
print(f"correlación Pearson (positividad, mortalidad) = {r2:.3f}")
ultimo[["state", "positivity", "death_rate"]].sort_values("positivity", ascending=False).head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(
    data=ultimo, x="positivity", y="death_rate", ax=ax,
    scatter_kws={"alpha": 0.7}, line_kws={"color": "#E45756"},
)
ax.set_title("Positividad vs tasa de mortalidad por estado")
ax.set_xlabel("Positividad (positive / totalTestResults)")
ax.set_ylabel("Tasa de mortalidad (death / positive)")
fig.tight_layout()
fig.savefig(FIGURES / "bivariado_positividad_vs_mortalidad.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** Se observa una asociación positiva débil/moderada entre
positividad y mortalidad: los estados que detectan una mayor proporción de
positivos entre los testeados tienden a tener una mayor tasa de mortalidad
entre los casos. Esto es consistente con la hipótesis de la Pregunta 2, pero
no la demuestra: ambas métricas dependen del reporting y del acceso a tests.

**Limitación:** correlación observacional; no se puede inferir causalidad.


### 7.3 Casos diarios por estado (distribución)

**Pregunta:** ¿qué estados tienen días con picos extremos de casos nuevos?
**Método:** boxplot de `positiveIncrease` para los 8 estados con más casos
acumulados.


In [ ]:
top_states = ultimo.nlargest(8, "positive")["state"].tolist()
subset = clean[clean["state"].isin(top_states)]
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=subset, x="state", y="positiveIncrease", ax=ax, showfliers=False)
ax.set_title("Distribución de casos diarios nuevos · 8 estados con más casos")
ax.set_xlabel("Estado")
ax.set_ylabel("Casos nuevos por día")
fig.tight_layout()
fig.savefig(FIGURES / "bivariado_boxplot_estados.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** Los 8 estados con más casos acumulados también tienen la
mayor variabilidad diaria y los picos más altos. La mediana diaria de
California y Texas supera a la de Nueva York, aunque Nueva York concentró
más casos al inicio de la pandemia.

**Limitación:** se comparan distribuciones sin normalizar por población.


## 8. Tendencias temporales


### 8.1 Serie diaria de casos nuevos

**Pregunta:** ¿cómo evolucionan los casos diarios y qué papel tiene el fin de
semana?
**Método:** suma de `positiveIncrease` por fecha y media móvil de 7 días.


In [ ]:
diario = (
    clean.groupby("date", as_index=False)["positiveIncrease"].sum()
    .sort_values("date")
    .reset_index(drop=True)
)
diario["ma7"] = diario["positiveIncrease"].rolling(7, min_periods=1).mean()
print("días:", len(diario))
print("pico diario:", int(diario["positiveIncrease"].max()), "el", diario.loc[diario["positiveIncrease"].idxmax(), "date"].date())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(diario["date"], diario["positiveIncrease"], color="#A0CBE8", alpha=0.6, label="Casos nuevos (diario)")
ax.plot(diario["date"], diario["ma7"], color="#E45756", linewidth=2, label="Media móvil 7 días")
ax.set_title("Casos diarios nuevos a nivel nacional")
ax.set_xlabel("Fecha")
ax.set_ylabel("Casos nuevos por día")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "tendencia_casos_diarios.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
por_dia = clean.copy()
por_dia["weekday"] = por_dia["date"].dt.day_name()
orden = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
media_por_dia = (
    por_dia.groupby("weekday")["positiveIncrease"].mean().reindex(orden)
)
print(media_por_dia.round(0))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
media_por_dia.plot.bar(ax=ax, color="#4C78A8")
ax.set_title("Media de casos nuevos por día de la semana")
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Casos nuevos (media)")
fig.tight_layout()
fig.savefig(FIGURES / "tendencia_por_dia_semana.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** La serie diaria muestra varias olas claras a lo largo del
periodo. La media móvil de 7 días suaviza el ruido y deja ver la tendencia
real. El gráfico por día de la semana confirma la hipótesis: sábado y
domingo tienen una media mucho menor que los días laborables, por menor
reporting.

**Limitación:** la media móvil suaviza picos reales y no predice. El efecto
fin de semana es un artefacto de reporting, no una caída real de contagios.


### 8.1 Serie diaria de casos nuevos

**Pregunta:** ¿cómo evolucionan los casos diarios y qué papel tiene el fin de
semana?
**Método:** suma de `positiveIncrease` por fecha y media móvil de 7 días.


In [ ]:
diario = (
    clean.groupby("date", as_index=False)["positiveIncrease"].sum()
    .sort_values("date")
    .reset_index(drop=True)
)
diario["ma7"] = diario["positiveIncrease"].rolling(7, min_periods=1).mean()
print("días:", len(diario))
print("pico diario:", int(diario["positiveIncrease"].max()), "el", diario.loc[diario["positiveIncrease"].idxmax(), "date"].date())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(diario["date"], diario["positiveIncrease"], color="#A0CBE8", alpha=0.6, label="Casos nuevos (diario)")
ax.plot(diario["date"], diario["ma7"], color="#E45756", linewidth=2, label="Media móvil 7 días")
ax.set_title("Casos diarios nuevos a nivel nacional")
ax.set_xlabel("Fecha")
ax.set_ylabel("Casos nuevos por día")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "tendencia_casos_diarios.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
por_dia = clean.copy()
por_dia["weekday"] = por_dia["date"].dt.day_name()
orden = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
media_por_dia = (
    por_dia.groupby("weekday")["positiveIncrease"].mean().reindex(orden)
)
print(media_por_dia.round(0))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
media_por_dia.plot.bar(ax=ax, color="#4C78A8")
ax.set_title("Media de casos nuevos por día de la semana")
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Casos nuevos (media)")
fig.tight_layout()
fig.savefig(FIGURES / "tendencia_por_dia_semana.png", dpi=150, bbox_inches="tight")
plt.show()


**Lectura.** La serie diaria muestra varias olas claras a lo largo del
periodo. La media móvil de 7 días suaviza el ruido y deja ver la tendencia
real. El gráfico por día de la semana confirma la hipótesis: sábado y
domingo tienen una media mucho menor que los días laborables, por menor
reporting.

**Limitación:** la media móvil suaviza picos reales y no predice. El efecto
fin de semana es un artefacto de reporting, no una caída real de contagios.


## 9. Limitaciones

- **Casos reportados, no reales.** El subregistro varía por estado y por
  semana; no se puede inferir la incidencia real.
- **`totalTestResults` cambió de definición** a lo largo del periodo y entre
  estados; las comparaciones de positividad no son estrictamente homogéneas.
- **Los counts no son comparables entre estados** sin normalizar por
  población (se aborda en N3).
- **Fines de semana y festivos** bajan el reporting; la media móvil de 7
  días es la lectura correcta de la tendencia.
- **Correlación no implica causalidad.** Las asociaciones observadas (tests
  vs positivos, positividad vs mortalidad) no demuestran causa-efecto.
- **`dataQualityGrade` está vacía al 100 %** en esta copia, así que no se
  puede evaluar la calidad declarada del dato.
- **La población no está integrada aún**; los ratios por 100k son trabajo de
  N3.


## 10. Conclusiones

Las conclusiones responden a las 3 preguntas de la sección 2 con cifras de
las celdas anteriores.

### Pregunta 1 · ¿Qué estados concentran más casos y muertes acumuladas?

- **Respuesta:** unos pocos estados concentran la mayor parte de los casos y
  de las muertes. Los 10 primeros por casos acumulados suponen una fracción
  muy grande del total (ver tabla de la sección 6.1).
- **Evidencia:** `univariado_casos_acumulados.png` y
  `univariado_muertes_acumuladas.png`.
- **Limitación:** counts absolutos; el ranking cambia al normalizar por
  población (N3).

### Pregunta 2 · ¿La positividad se asocia con la mortalidad?

- **Respuesta:** se observa una asociación positiva entre positividad y tasa
  de mortalidad por estado (ver sección 7.2), consistente con la hipótesis,
  pero débil/moderada.
- **Evidencia:** `bivariado_positividad_vs_mortalidad.png` y el coeficiente
  de correlación reportado.
- **Limitación:** correlación observacional; no se puede inferir causalidad.

### Pregunta 3 · ¿Cómo evolucionan los casos diarios y qué papel tiene el fin de semana?

- **Respuesta:** la serie nacional muestra varias olas claras y caídas
  sistemáticas en sábado y domingo por menor reporting; la media móvil de 7
  días deja ver la tendencia real.
- **Evidencia:** `tendencia_casos_diarios.png` y
  `tendencia_por_dia_semana.png`.
- **Limitación:** la media móvil suaviza picos reales y no predice.

### Próximos pasos

- Normalizar por población y calcular ratios por 100k (N3-01, N3-02).
- Detectar y tratar incrementos negativos y generar la auditoría completa
  (N3-03).
- Segmentar por región y por olas (N3-05).
- Escribir el informe ejecutivo y el README (N2-09, N2-10).
